In [1]:
!pip -q install evaluate jiwer soundfile librosa torchcodec

In [2]:
import torch
import evaluate
import numpy as np
from torch.utils.data import DataLoader
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration, set_seed

set_seed(42)

### Load raw LibriSpeech splits

In [3]:
train_raw = load_dataset("openslr/librispeech_asr", "clean", split="train.100[:1000]")
val_raw = load_dataset("openslr/librispeech_asr", "clean", split="validation[:100]")
test_raw = load_dataset("openslr/librispeech_asr", "clean", split="test[:100]")


train_raw = train_raw.shuffle(seed=42)

print("Raw Train:", len(train_raw))
print("Raw Val  :", len(val_raw))
print("Raw Test :", len(test_raw))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

clean/train.360/0038.parquet:   0%|          | 0.00/506M [00:00<?, ?B/s]

clean/train.360/0040.parquet:   0%|          | 0.00/480M [00:00<?, ?B/s]

...

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Raw Train: 1000
Raw Val  : 100
Raw Test : 100


_Inspect one raw audio sample_

In [4]:
def get_audio_array_and_rate(audio):
    if hasattr(audio, "get_all_samples"):
        samples = audio.get_all_samples()
        array = samples.data.squeeze(0).numpy()
        sampling_rate = samples.sample_rate
    else:
        array = audio["array"]
        sampling_rate = audio["sampling_rate"]
    return array, sampling_rate


sample = train_raw[0]
audio_array, sampling_rate = get_audio_array_and_rate(sample["audio"])


print("Audio length:", len(audio_array))
print("Sampling rate:", sampling_rate)
print("Text:", sample["text"])
print("First 20 samples:", audio_array[:20])

Audio length: 213760
Sampling rate: 16000
Text: NO PERSON MADE ANY ATTEMPT AT ACCOUNTING FOR HIS OPINION THAT HE WAS UNIQUE APPEARED SO UNDENIABLE THAT IT WAS DEEMED IMPERTINENT TO INQUIRE WHEREIN THE UNIQUITY CONSISTED
First 20 samples: [ 0.00350952  0.00134277 -0.00057983 -0.00216675 -0.00311279 -0.00369263
 -0.00421143 -0.00527954 -0.00637817 -0.00860596 -0.0116272  -0.01412964
 -0.01550293 -0.015625   -0.01516724 -0.01425171 -0.01409912 -0.01541138
 -0.01644897 -0.01773071]
